# B2 · Choosing the threshold with the real costs

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Given a cost for each kind of error (an unpaid loan costs 10, a good customer turned down costs 1), choose the decision threshold **using only the training set**
(out-of-sample predictions with `cross_val_predict`), compare it with the theoretical threshold $c_{FP}/(c_{FP}+c_{FN})$ and with `class_weight="balanced"`, and measure the cost on the test set.

## Dataset

`ds.loans(with_missing=False)` with `ratio = amount/income`: features `ratio, age, job_tenure, income`, stratified 70/30 split. Model: logistic with scaler.

## Educational purpose

Take classification from the metric to **money**: threshold 0.5 is not a neutral default but an implicit choice that assumes equal costs. You will learn
a clean procedure (threshold chosen on the training set, test set touched once) and the theoretical rule for calibrated probabilities.

## Task

1. Define `cost(y, p, threshold)`: total cost if you reject (`p >= threshold`) and do not grant the loan. A defaulter accepted (false negative) costs 10; a non-defaulter turned down (false positive) costs 1.
2. Get with `cross_val_predict(model, X_tr, y_tr, cv=cv, method="predict_proba")` the **out-of-sample** probabilities on the training set and compute `oof_costs`: the cost for each threshold of `thresholds` (already defined). Save `best_threshold` (minimum-cost threshold) and `theoretical_threshold = 1/11`.
3. On the **test set** (model trained on the training set) compute `test_costs`, a dictionary with: `threshold 0.5`, `best threshold`, `theoretical threshold`, `class_weight balanced (0.5)`, `approve all`, `reject all`.
4. Save `saving = 1 − cost(best threshold) / cost(threshold 0.5)` on the test set.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
df["ratio"] = df["amount"] / df["income"]
cols = ["ratio", "age", "job_tenure", "income"]
X_tr, X_te, y_tr, y_te = train_test_split(df[cols], df["defaulted"], test_size=0.3, stratify=df["defaulted"], random_state=0)
y_tr, y_te = y_tr.to_numpy(), y_te.to_numpy()
cv = StratifiedKFold(5, shuffle=True, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

## Your work

In [ ]:
def cost(y, p, threshold):
    ...

In [ ]:
oof = None
oof_costs = None
best_threshold = None
theoretical_threshold = None

In [ ]:
test_costs = {}
saving = None
pd.Series(test_costs)

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`cost`: `pred = p >= threshold; 10 * ((y == 1) & ~pred).sum() + 1 * ((y == 0) & pred).sum()`. Best threshold: `thresholds[np.argmin(oof_costs)]`. For `balanced` train a second model with `LogisticRegression(class_weight="balanced")` and use threshold 0.5. "Approve all" = threshold 1.1 (nobody rejected).'):
    check(cost(np.array([1, 0]), np.array([0.2, 0.9]), 0.5) == 11, "cost: a defaulter not caught (10) + a good one turned down (1)", "FN weighs 10, FP weighs 1")
    check(oof is not None and oof.shape == (len(y_tr),) and len(oof_costs) == len(thresholds), "out-of-sample predictions and one cost per threshold", "predict_proba[:, 1] from cross_val_predict")
    check(0.05 <= best_threshold <= 0.2 and abs(best_threshold - theoretical_threshold) < 0.06, "the best threshold is close to the theoretical 1/11 (about 0.09)", "with calibrated probabilities the best threshold is c_FP / (c_FP + c_FN)")
    check(test_costs["best threshold"] < 0.7 * test_costs["threshold 0.5"], "with the best threshold the cost drops by over 30% compared to 0.5", "use the threshold found on the training set, applied to the test set")
    check(test_costs["best threshold"] < test_costs["class_weight balanced (0.5)"] and test_costs["best threshold"] < min(test_costs["approve all"], test_costs["reject all"]),
          "it beats class_weight balanced and the two trivial policies", "compare the six costs")
    check(close(saving, 1 - test_costs["best threshold"] / test_costs["threshold 0.5"]) and saving > 0.3, "correct saving", "1 - best cost / cost at 0.5")
    check(test_costs["threshold 0.5"] > test_costs["reject all"], "at threshold 0.5 the model costs more than rejecting everyone: the default threshold can lose value", "compare threshold 0.5 and reject all")
done()

---
The commented solution is in `solutions/B_with_sklearn/B2_optimal_threshold_with_costs.ipynb`. Open it after you pass the check, or after you have really tried.